In [1]:
from netgen.occ import *
from netgen.geom2d import SplineGeometry
from ngsolve import *
from ngsolve.solvers import *
from ngsolve.webgui import Draw
from netgen.webgui import Draw as DrawGeo

import numpy as np

In [11]:
# Define Geometry
rod = MoveTo(-0.025, 0).Rectangle(0.05, 0.25).Face()
rod.edges.Min(Y).name = "bottom"

head = Circle((0, 0.25), 0.075).Face()

geo = rod + head

geo = geo.Rotate(Axis((0,0,0),Z),-1)

geo.name = "pendulum"

mesh = Mesh(OCCGeometry(geo, dim=2).GenerateMesh(maxh=0.025))

mesh.Curve(4)
Draw (mesh);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [12]:
mesh.GetMaterials()


('pendulum',)

In [3]:
V = VectorH1(mesh, order=2)
Q = NumberSpace(mesh, definedon=mesh.Boundaries('bottom'))
fes = V * Q**2
(u,q), (v, p) = fes.TnT()

gfut = GridFunction(V, multidim=0)

# define the needed GridFunctiond required in the scheme
gfu = GridFunction(fes)
gfv = GridFunction(fes)
gfa = GridFunction(fes)

gfuold = GridFunction(fes)
gfvold = GridFunction(fes)
gfaold = GridFunction(fes)

**Set initial angular position**

In [13]:
from math import cos, sin, pi
theta0 = 45
theta = theta0*pi/180.0
c, s = cos(theta), sin(theta)

# rotation pivot (e.g., hole center or origin)
cx, cy = 0.0, 0.0

xr = x - cx
yr = y - cy

# displacement for pure rotation (no scaling)
u_rot = CF( ( (c-1.0)*xr - s*yr,
              s*xr      + (c-1.0)*yr ) )

gfu.components[0].Set(u_rot, definedon=mesh.Materials("pendulum"))
gfuold.vec[:] = gfu.vec

Draw(gfu.components[0], mesh, "displacement", deformation=True)


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

**Set initial angular velocity**

In [7]:
cx, cy = 0.0, 0.0

omega0 = 0 # initial angular velocity (rad/s)

v0 = CF( (-omega0*(y-cy), omega0*(x-cx)) )

gfv.components[0].Set(v0)
gfvold.components[0].Set(v0)

**Set initial angular acceleration**

In [8]:
alpha0 = 1 # initial angular acceleration (rad/s²)
a0 = CF( (-alpha0*(y-cy), -alpha0*(x-cx)) )

gfa.components[0].Set(a0)
gfaold.components[0].Set(a0)

**Draw Initial States of Grid Functions**

In [9]:
Draw(gfu.components[0], mesh, "displacement", deformation=True)


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [ ]:
Draw(gfv.components[0], mesh, "velocity", deformation=False, vectors=True)


In [ ]:
Draw(gfa.components[0], mesh, "acceleration", deformation=False, vectors=True)

In [ ]:
def C(u):
    F = Id(2) + Grad(u)
    return F.trans * F

E, nu = 2100, 0.2

mu = E / 2 / (1+nu)
lam = E * nu / ((1+nu)*(1-2*nu))

def NeoHooke (C):
    return 0.5*mu*(Trace(C-Id(2)) + 2*mu/lam*Det(C)**(-lam/2/mu)-1)


bfa = BilinearForm(fes)
bfa += Variation(NeoHooke(C(u))*dx).Compile() 
bfa += (InnerProduct(u, p) + InnerProduct(v, q)) * ds('bottom') # we add the constraints

tau = 0.025  # time step size
tend = 10
rho = 7e3
force = CF((0, -9.81*0.1))  # gravity force


vel_new = 2/tau * (u-gfuold.components[0]) - gfvold.components[0]
acc_new = 2/tau * (vel_new-gfvold.components[0]) - gfaold.components[0]

# need to add to the bilinear form since it depends on the current valurs of the GridFunctions

bfa += acc_new*v*dx
bfa += -force*v*dx

In [ ]:
from ngsolve.solvers import Newton
from numba import njit

t = 0
i =1

@njit(parallel=True)
def solve():
    with TaskManager():
        while t < tend:
            i += 1
            t += tau
            Newton(a=bfa, u=gfu, printing=False, inverse="sparsecholesky")

        if i % 5 == 0:
            #scene.Redraw()
            gfut.AddMultiDimComponent(gfu.components[0].vec)
        gfv.vec[:] = 2/tau * (gfu.vec-gfuold.vec) - gfvold.vec
        gfa.vec[:] = 2/tau * (gfv.vec-gfvold.vec) - gfaold.vec

        gfuold.vec[:] = gfu.vec
        gfvold.vec[:] = gfv.vec
        gfaold.vec[:] = gfa.vec

In [ ]:
settings = {"Multidim": {
    "speed" : 2
}}


Draw(gfut, mesh,
     interpolate_multidim=True,
     deformation = True, animate=True,
     autoscale = False,
     min = 0, max = 0.70,
     settings = settings);

**Linear Elasticity**

In [21]:
V = VectorH1(mesh, order=2)
Q = NumberSpace(mesh, definedon=mesh.Boundaries('bottom'))
fes = V * Q**2
(u,q), (v, p) = fes.TnT()

gfut = GridFunction(V, multidim=0)

# define the needed GridFunctiond required in the scheme
gfu = GridFunction(fes)
gfv = GridFunction(fes)
gfa = GridFunction(fes)

gfuold = GridFunction(fes)
gfvold = GridFunction(fes)
gfaold = GridFunction(fes)

from math import cos, sin, pi
theta0 = -135
theta = theta0*pi/180.0
c, s = cos(theta), sin(theta)

# rotation pivot (e.g., hole center or origin)
cx, cy = 0.0, 0.0

xr = x - cx
yr = y - cy

# displacement for pure rotation (no scaling)
u_rot = CF( ( (c-1.0)*xr - s*yr,
              s*xr      + (c-1.0)*yr ) )

gfu.components[0].Set(u_rot)
gfuold.vec[:] = gfu.vec


# --- Material & thickness (SI) ---
E   = 2.1e11           # [Pa]
nu  = 0.30
t   = 0.01             # [m] thickness
rho = 7850.0           # [kg/m^3]
rhoA = rho*t
mode = "plane_stress"  # or "plane_strain"

# --- Constitutive ---
def eps(u): return Sym(Grad(u))
def lame_from_E_nu(E, nu, mode="plane_strain"):
    mu = E/(2*(1+nu))
    if mode == "plane_strain":
        lam = E*nu/((1+nu)*(1-2*nu))
    elif mode == "plane_stress":
        lam = 2*mu*nu/(1-nu)   # = E*nu/(1-nu**2)
    else:
        raise ValueError
    return lam, mu
lam, mu = lame_from_E_nu(E, nu, mode)

def lin_elastic_energy(u):
    e = eps(u)
    return 0.5*lam*Trace(e)**2 + mu*InnerProduct(e, e)

tau = 0.025 / sqrt(E/rho)
tau = 1e-3
print("time step size tau =", tau)

# --- Bilinear form ---
bfa = BilinearForm(fes)
bfa += Variation(lin_elastic_energy(u) * dx).Compile()
bfa += (InnerProduct(u, p) + InnerProduct(v, q)) * ds('bottom')

# --- Dynamics (your scheme) ---
vel_new = 2/tau * (u - gfuold.components[0]) - gfvold.components[0]
acc_new = 2/tau * (vel_new - gfvold.components[0]) - gfaold.components[0]

g = -9.81
bfa += rhoA * InnerProduct(acc_new, v) * dx      # inertia with areal density
bfa += -rhoA * CF((0, g)) * v * dx               # gravity downward

from ngsolve.solvers import Newton

tend = 1
t = 0
i =1



time step size tau = 0.001


In [22]:
Draw(gfu.components[0], mesh, "displacement", deformation=True)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [23]:
def solve():
    tend = 1
    t = 0
    i =1
    with TaskManager():
        while t < tend:
            i += 1
            t += tau
            Newton(a=bfa, u=gfu, printing=False, inverse="sparsecholesky")

            if  i % 50 == 0:
                print("t =", t)

            if i % 5 == 0:
                #scene.Redraw()
                gfut.AddMultiDimComponent(gfu.components[0].vec)
            gfv.vec[:] = 2/tau * (gfu.vec-gfuold.vec) - gfvold.vec
            gfa.vec[:] = 2/tau * (gfv.vec-gfvold.vec) - gfaold.vec

            gfuold.vec[:] = gfu.vec
            gfvold.vec[:] = gfv.vec
            gfaold.vec[:] = gfa.vec

solve()

t = 0.04900000000000004
t = 0.09900000000000007
t = 0.1490000000000001
t = 0.19900000000000015
t = 0.2490000000000002
t = 0.2990000000000002
t = 0.34900000000000025
t = 0.3990000000000003
t = 0.44900000000000034
t = 0.4990000000000004
t = 0.5490000000000004
t = 0.5990000000000004
t = 0.6490000000000005
t = 0.6990000000000005
t = 0.7490000000000006
t = 0.7990000000000006
t = 0.8490000000000006
t = 0.8990000000000007
t = 0.9490000000000007
t = 0.9990000000000008


In [24]:
settings = {"Multidim": {
    "speed" : 1
}}


Draw(gfut, mesh,
     interpolate_multidim=True,
     deformation = True, animate=True,
     autoscale = False,
     min = 0, max = 0.70,
     settings = settings);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Multidim': {'speed': 1}}, '…

---------------------






































































































## 3D Pendulum

In [ ]:
# Define Parameters
length = 0.35
r_rod = 0.025
r_head = 0.075

# Define Points
base_rod_pt = (0, 0, 0)
top_rod_pt = (0, length,0)


rod = Cylinder(base_rod_pt, Y, r_rod, length)
rod.faces.Min(Y).name = "base"

head = Sphere(top_rod_pt, r_head)
vec = gp_Vec(0, length, 0)
head.Move(vec)

shape = rod + head

mesh = Mesh(OCCGeometry(shape, dim=3).GenerateMesh(maxh=2))
mesh.Curve(3)
Draw(mesh)

In [ ]:
def C_3D(u):
    F = Id(3) + Grad(u)
    return F.trans * F

def NeoHooke_3D (C):
    return 0.5*mu*(Trace(C-Id(3)) + 2*mu/lam*Det(C)**(-lam/2/mu)-1)

V = VectorH1(mesh, order=2)
Q = NumberSpace(mesh, definedon=mesh.Boundaries('base'))

fes = V * Q**3
(u,q), (v, p) = fes.TnT()

gfut = GridFunction(V, multidim=0)

# define the needed GridFunctiond required in the scheme
gfu = GridFunction(fes)
gfv = GridFunction(fes)
gfa = GridFunction(fes)

gfuold = GridFunction(fes)
gfvold = GridFunction(fes)
gfaold = GridFunction(fes)

bfa = BilinearForm(fes)
bfa += Variation(NeoHooke_3D(C_3D(u))*dx).Compile() 
bfa += (InnerProduct(u, p) + InnerProduct(v, q)) * ds('base') # we add the constraints

tau = 0.025  # time step size
tend = 10
rho = 7e3
force = CF((0, -1, 0))  # gravity force

In [ ]:
from math import cos, sin, pi

theta = -45.0*pi/180.0
c, s = cos(theta), sin(theta)

# rotation pivot (e.g., hole center or origin)
cx, cy, cz = 0.0, 0.0, 0.0

xr = x - cx
yr = y - cy
zr = z - cz

# displacement for pure rotation (no scaling)
u_rot = CF( ( (c-1.0)*xr - s*yr,
              s*xr      + (c-1.0)*yr,
              0.0 ) )


gfu.components[0].Set(u_rot)
gfuold.vec[:] = gfu.vec

In [ ]:
vel_new = 2/tau * (u-gfuold.components[0]) - gfvold.components[0]
acc_new = 2/tau * (vel_new-gfvold.components[0]) - gfaold.components[0]

# need to add to the bilinear form since it depends on the current valurs of the GridFunctions

bfa += acc_new*v*dx
bfa += -force*v*dx

In [ ]:
gfut.AddMultiDimComponent(gfu.components[0].vec)
scene = Draw(gfu.components[0], mesh, "deformation", deformation=True)  

In [ ]:
tau = 0.025  # time step size
tend = 0.75
rho = 7e3
force = CF((0, -1, 0))  # gravity force


vel_new = 2/tau * (u-gfuold.components[0]) - gfvold.components[0]
acc_new = 2/tau * (vel_new-gfvold.components[0]) - gfaold.components[0]

# need to add to the bilinear form since it depends on the current valurs of the GridFunctions

bfa += acc_new*v*dx
bfa += -force*v*dx

In [ ]:
from ngsolve.solvers import Newton
gfut.AddMultiDimComponent(gfu.components[0].vec)
t = 0
i =1
with TaskManager():
    while t < tend:
        i += 1
        t += tau
        Newton(a=bfa, u=gfu, printing=False, inverse="sparsecholesky")

        if i % 5 == 0:
            #scene.Redraw()
            gfut.AddMultiDimComponent(gfu.components[0].vec)
        gfv.vec[:] = 2/tau * (gfu.vec-gfuold.vec) - gfvold.vec
        gfa.vec[:] = 2/tau * (gfv.vec-gfvold.vec) - gfaold.vec

        gfuold.vec[:] = gfu.vec
        gfvold.vec[:] = gfv.vec
        gfaold.vec[:] = gfa.vec

In [ ]:
settings = {"Multidim": {
    "speed" : 1
}}

Draw(gfut, mesh,
     interpolate_multidim=True,
     deformation = True,
     animate=True,
     autoscale = False,
     min = 0, max = 2,
     settings = settings);